# 1. INTRODUCTION

Shazam works by creating a 'constellation map' of local energy maxima in the spectrogram of an audio signal. 
Because peak amplitudes survive intense ambient noise and standard audio compression, this geometric method yields incredible robustness.

## High-Level Flow
1. **Spectrogram Generation**: Convert raw time-domain audio to the time-frequency domain.
2. **Peak Detection**: Filter the spectrogram to isolate statistically significant local maxima (peaks).
3. **Fingerprinting (Hashing)**: Group peaks combinatorially. A target peak and an anchor peak with a time delta $\Delta t$ form a hash: `(freq1, freq2, delta_t)`.
4. **Database Matching**: Store these hashes. When a query comes in, look up the hashes, and use histogram voting to see if they perfectly align via a consistent time offset (`db_offset - query_offset`).

# 2. AUDIO PREPROCESSING
Raw audio comes in various formats and sample rates. To ensure strict equivalency:
1. **Downmix stereo to mono**: The spatial panning doesn't help identification.
2. **Resample**: Fixed target of 22050 Hz caps our Nyquist limit at ~11kHz. 
3. **Normalize**: Amplitude boundaries are strictly float [-1.0, 1.0].

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import warnings
import scipy.ndimage as ndimage
from collections import defaultdict
import pickle

def load_audio(file_path: str, target_sr: int = 22050, duration: float = None, offset: float = 0.0):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            audio, sr = librosa.load(file_path, sr=target_sr, mono=True, offset=offset, duration=duration)
            
        if audio.size > 0:
            max_val = np.max(np.abs(audio))
            if max_val > 0:
                audio = audio / max_val
        return audio, sr
    except Exception as e:
        raise RuntimeError(f"Failed to load audio: {str(e)}")

# Generating a synthetic ground truth for validation
samplerate = 22050
duration = 8.0
t = np.linspace(0, duration, int(samplerate * duration), endpoint=False)
synthetic_audio = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 880 * t)

plt.figure(figsize=(10, 3))
plt.plot(t[:1000], synthetic_audio[:1000]) 
plt.title("Synthetic Waveform Snippet")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()

# 3. SPECTROGRAM GENERATION
We compute the Short-Time Fourier Transform (STFT) from scratch using NumPy. 
We use a Hanning window to prevent spectral leakage at the frame edges, and a 50% overlap (`hop_length`) to capture transients accurately.

In [ ]:
def compute_stft(audio: np.ndarray, n_fft: int = 2048, hop_length: int = 1024) -> np.ndarray:
    window = np.hanning(n_fft)
    pad_len = n_fft // 2
    padded_audio = np.pad(audio, pad_len, mode='constant')
    
    num_frames = 1 + (len(padded_audio) - n_fft) // hop_length
    stft_matrix = np.zeros((1 + n_fft // 2, num_frames), dtype=np.complex64)
    
    for i in range(num_frames):
        start = i * hop_length
        end = start + n_fft
        frame = padded_audio[start:end] * window
        stft_matrix[:, i] = np.fft.rfft(frame, n=n_fft)
    return stft_matrix

def get_spectrogram(stft_matrix: np.ndarray, ref: float = 1.0, amin: float = 1e-10) -> np.ndarray:
    magnitude = np.abs(stft_matrix)
    return 20 * np.log10(np.maximum(amin, magnitude) / ref)

n_fft = 2048
hop_length = 1024
stft_matrix = compute_stft(synthetic_audio, n_fft=n_fft, hop_length=hop_length)
spec_db = get_spectrogram(stft_matrix)

plt.figure(figsize=(10, 4))
plt.imshow(spec_db, aspect='auto', origin='lower', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram (Log Magnitude)')
plt.ylim(0, 150)
plt.show()

# 4. PEAK DETECTION
We detect local maxima in the time-frequency domain using a 2D neighborhood maximum filter. This ensures we survive massive volume changes because we're looking for relative peaks, not absolute ones.

In [ ]:
def find_peaks(spectrogram_db: np.ndarray, fp_dim: int = 15, time_dim: int = 15, threshold: float = 10.0):
    neighborhood = np.ones((fp_dim, time_dim), dtype=bool)
    local_max = ndimage.maximum_filter(spectrogram_db, footprint=neighborhood) == spectrogram_db
    background = (spectrogram_db > threshold)
    detected_peaks = local_max & background
    
    freq_idx, time_idx = np.where(detected_peaks)
    peaks = list(zip(freq_idx, time_idx))
    peaks.sort(key=lambda x: x[1])
    return peaks

peaks = find_peaks(spec_db, threshold=0.0)
freqs, times = zip(*peaks)

plt.figure(figsize=(10, 4))
plt.imshow(spec_db, aspect='auto', origin='lower', cmap='magma')
plt.scatter(times, freqs, c='cyan', s=10, marker='x', alpha=0.8)
plt.title('Constellation Map: Peaks Overlaid')
plt.ylim(0, 150)
plt.show()

# 5. FINGERPRINT GENERATION (CORE PART)
Peaks alone shift if the queried audio snippet starts 1 second into the song. 
**Combinatorial Pairs**: We pair an anchor peak with target peaks in a defined window ahead of it. `(f_anchor, f_target, Delta_Time)`.
Because `Delta_Time` is relative to the anchor, this hash footprint is 100% time-invariant!

In [ ]:
def generate_hashes(peaks: list[tuple[int, int]], target_zone_fw: int = 200, target_zone_freq: int = 200, fan_out: int = 10, min_time_delta: int = 1):
    hashes = []
    num_peaks = len(peaks)
    
    for i in range(num_peaks):
        freq1, time1 = peaks[i]
        for j in range(1, fan_out + 1):
            if i + j < num_peaks:
                freq2, time2 = peaks[i + j]
                delta_t = time2 - time1
                
                if min_time_delta <= delta_t <= target_zone_fw:
                    if abs(freq2 - freq1) <= target_zone_freq:
                        # Pack into a beautiful single 32-bit uint
                        h = ((freq1 & 0x7FF) << 21) | ((freq2 & 0x7FF) << 10) | (delta_t & 0x3FF)
                        hashes.append((h, time1))
    return hashes

hashes = generate_hashes(peaks)
print(f"Total peaks identified: {len(peaks)}")
print(f"Combinatorial hashes extracted: {len(hashes)}")
print(f"Sample hash (uint32, absolute_time): {hashes[0]}")

# 6. DATABASE DESIGN
In-memory inverted index architecture: `HashKey -> List of (SongID, absoluteOffset)`.

In [ ]:
class SongDatabase:
    def __init__(self):
        self.hashes = defaultdict(list)
        self.songs = {}
        self.next_song_id = 1
        
    def add_song(self, song_name: str, hashes: list[tuple[int, int]]) -> int:
        song_id = self.next_song_id
        self.next_song_id += 1
        self.songs[song_id] = song_name
        for h_val, time_offset in hashes:
            self.hashes[h_val].append((song_id, time_offset))
        return song_id
        
    def query(self, h_val: int):
        return self.hashes.get(h_val, [])

db = SongDatabase()
song_id = db.add_song("Synthetic A4/A5", hashes)
print(f"Stored {len(hashes)} footprint combinations in RAM for Song ID: {song_id}")

# 7. QUERY MATCHING
Given a cropped/noisy query snippet, we extract its hashes and lookup the DB.
The magic is **Histogram Voting**: True matches will naturally group on a single consistent time Delta: `db_offset - query_offset = lag_constant`. Random collisions will scatter thinly.

In [ ]:
def match_hashes(query_hashes: list[tuple[int, int]], db: SongDatabase):
    song_scores = defaultdict(lambda: defaultdict(int))
    for h_val, query_offset in query_hashes:
        matches = db.query(h_val)
        for (song_id, db_offset) in matches:
            dt = db_offset - query_offset
            song_scores[song_id][dt] += 1
            
    if not song_scores:
        return None, None, 0
        
    best_song_id = None
    best_offset = None
    max_votes = 0
    
    for song_id, dt_hist in song_scores.items():
        best_dt = max(dt_hist.keys(), key=lambda t: dt_hist[t])
        votes = dt_hist[best_dt]
        if votes > max_votes:
            max_votes = votes
            best_song_id = song_id
            best_offset = best_dt
            
    return best_song_id, best_offset, max_votes

# 8. PERFORMANCE OPTIMIZATION
- **Vectorization**: Hanning dot products inside STFT loops vs direct python lists.
- **Bitpacking Hashes**: Mapping `(f1, f2, dt)` into a single 32-bit integer minimizes overhead massively.
- **Memory Maps**: At 10M songs, use `rocksdb` or `sqlite3` inverted disk-tables.
- **SIMD**: Leveraging compiled `C/C++` arrays natively in NumPy avoids Python's GIL constraints.

# 9. DEMO
Simulating a noisy real-world query via a cropped snippet with heavily injected 10dB Gaussian noise.

In [ ]:
import random

# Crop seconds 3.0 to 5.0
query_audio = synthetic_audio[int(3.0 * samplerate):int(5.0 * samplerate)]

# Add heavy white noise
noise_power = 0.5 / (10 ** (10 / 10))
noisy_query = query_audio + np.random.normal(scale=np.sqrt(noise_power), size=len(query_audio))

# Engine Pipeline applied to query
q_stft = compute_stft(noisy_query)
q_spec = get_spectrogram(q_stft)
q_peaks = find_peaks(q_spec, threshold=0)
q_hashes = generate_hashes(q_peaks)

best_id, match_offset, confidence = match_hashes(q_hashes, db)

print(f"--- MATCH RESULTS ---")
print(f"Query Hashes: {len(q_hashes)}")
print(f"Identified Song ID: {best_id} ({db.songs.get(best_id, 'Unknown')})")
print(f"Histogram Confidence Votes: {confidence}")
print(f"Time offset (frames): {match_offset} -> roughly {match_offset * 1024 / 22050:.2f} seconds lag")

# 10. ENGINEERING INSIGHTS
- **Why Constellations?**: Directly subtracting FFTs fails identically with minor noise or mic alignment. Peaks survive analog/digital crossover perfectly.
- **Limitations**: The algorithm assumes strict stationary playback speeds. Real Shazam natively tracks frequency *ratios* alongside $\Delta t$ to grant pitch-shift immunity.

# 11. HOW TO BUILD A SONG DATABASE (PRACTICAL GUIDE)
### Step-by-Step Architecture Expansion
1. **Ingest Pipeline Data Store**: Define an S3 bucket or high-IO NAS array holding master files.
2. **Bulk Extraction**: Map scripts (`compute_stft`, `find_peaks`, `generate`) natively over thousands of concurrent AWS Lambda or Ray workers.
3. **Storage Architect**: Dump `uint32` hashes into a horizontally partitioned NoSQL database (Cassandra or DynamoDB). Partition Key = Hash, Sort Key = SongID.
4. **Handling Scale Constraints**: For 100M songs, hashes will collide. Statistically, silence `(0,0,x)` will hotspot DB nodes. Implement dynamic "Silence Culling" and strict noise floors.